# Recurrent Energy NodeField ablation

Thin driver for the repository implementation. Default execution runs only the seed-0 smoke matrix (100 graphs, ten epochs). Set `RUN_FULL = True` to train five seeds on 1,000 graphs and run the complete grid. Every execution creates a new result directory. The driver uses CUDA when available, then Apple MPS, and otherwise CPU. The full CPU run is intended as an overnight batch rather than an interactive check; the saved environment records the selected accelerator.

**Predefined outcome:** feasible structural condition match over all attempts. A positive paired seed-level 95% interval and an absolute effect of at least 0.05 are the full-run evidence criterion. The single-seed smoke run cannot establish significance. State changes measure empirical stabilization, not mathematical convergence. The primary metric is computed from decoded graph structure; the training-split feasibility estimator is optional secondary diagnostics, not a held-out validity oracle.


## 0 — Experiment metadata

In [1]:
EXPERIMENT_NAME = "recurrent_energy_nodefield_ablation_v1"
SEEDS = [0, 1, 2, 3, 4]
RUN_FULL = False


## 1 — Imports and reproducibility
Repository helpers seed Python, NumPy and Torch. Unsupported deterministic operations emit warnings; environment, selected accelerator, library versions, and resolved configurations are saved with each run. The validation-calibrated anytime study samples its decoder-isomorphism check at the configured stride (16 by default) and always includes the full-budget step.

In [2]:
from conditional_node_field_graph_generator.extensions.demo.recurrent_experiments import (
    RecurrentExperiment, summarize_results, plot_results, load_results,
    analysis_section, decision_report,
)
experiment = RecurrentExperiment(smoke=not RUN_FULL)
print(experiment.run_dir)
K_TRAIN = experiment.configs["recurrent_energy_annealed"]["model"]["recurrent_training_steps"]


artifact/recurrent_nodefield/20260905T111715-2f96c4ec


## 2 — One canonical dataset
Use the existing cycle/path/star generator and fitted vectorizers. The cached 80/10/10 split, supervision and training-only preprocessing are shared by every model.

In [3]:
experiment.prepare()
print({name: len(indices) for name, indices in experiment.splits.items()})


Enabling RDKit 2026.03.4 jupyter extensions


{'train': 80, 'validation': 10, 'test': 10}


## 3 — Primary model matrix

| ID | Model | Memory | Corruption | Energy |
|---|---|---|---|---|
| A | Baseline | No | Existing fixed sigma | Yes |
| B | RENF | Yes | Constant | Yes |
| C | RENF | Yes | Annealed | Yes |
| D | Same checkpoint as C | Intervention-dependent | Annealed | Yes |

D is an alias, not another training run. Parameter counts are measured after setup. Raw RENF has extra parameters; the notebook does not claim parameter matching.


In [4]:
print({name: config["model"] for name, config in experiment.configs.items()})


{'baseline': {'latent_embedding_dimension': 64, 'number_of_transformer_layers': 1, 'transformer_attention_head_count': 4, 'transformer_dropout': 0.1, 'batch_size': 16, 'learning_rate': 0.0001, 'maximum_epochs': 250, 'early_stopping_patience': 20, 'early_stopping_min_delta': 0.0, 'node_field_sigma': 0.2, 'sampling_step_size': 0.05, 'langevin_noise_scale': 0.0, 'recurrent_training_steps': 8, 'recurrent_hidden_dimension': 64, 'recurrent_detach_interval': 4, 'recurrent_update_scale': 1.0, 'recurrent_initial_state': 'zeros', 'recurrent_state_normalization': True, 'recurrent_sigma_min': 0.02, 'recurrent_sigma_max': 0.2, 'recurrent_supervise_all_steps': True, 'recurrent_loss_discount': 1.0, 'node_embedding_svd_dimension': 32, 'graph_embedding_svd_dimension': 32, 'node_vectorizer_parallel': False, 'graph_vectorizer_parallel': False, 'feasibility_parallel': False, 'decoder_n_jobs': 1, 'decoder_solver_threads': 1, 'locality_horizon': 2, 'use_feasibility_filtering': False, 'feasibility_oracle_can

## 4 — Sanity checks before training
Abort on nonfinite loss or gradients. Check shapes, padding, hidden influence and score gradients. Parameter-sharing and finite-difference checks are covered by the prerequisite test suite.

In [5]:
experiment.sanity_checks()


{'model': 'baseline', 'loss': 32.15438461303711, 'score_norm': 25.542808532714844, 'hidden_norm': 0.0, 'gradient_norm': 4.0426459312438965, 'parameter_count': 175829, 'peak_gpu_memory': None}


/Users/f.costa/Code/NodeField/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


{'model': 'recurrent_energy_constant', 'loss': 32.20203399658203, 'score_norm': 19.231800079345703, 'hidden_norm': 0.9135572910308838, 'gradient_norm': 6.667080402374268, 'parameter_count': 196949, 'peak_gpu_memory': None}


/Users/f.costa/Code/NodeField/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


{'model': 'recurrent_energy_annealed', 'loss': 645.0696411132812, 'score_norm': 19.231800079345703, 'hidden_norm': 0.9135572910308838, 'gradient_norm': 8.115541458129883, 'parameter_count': 196949, 'peak_gpu_memory': None}


/Users/f.costa/Code/NodeField/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


,model,loss,score_norm,hidden_norm,gradient_norm,parameter_count,peak_gpu_memory
0,baseline,32.154385,25.542809,0.000000,4.042646,175829,None
1,recurrent_energy_constant,32.202034,19.231800,0.913557,6.667080,196949,None
2,recurrent_energy_annealed,645.069641,19.231800,0.913557,8.115541,196949,None


## 5 — Train and retain checkpoints
Smoke: equal ten-epoch budgets. Full: identical validation selection and patience; all epoch checkpoints are kept for matched-update curriculum comparisons. No test data selects checkpoints.

In [6]:
experiment.train()


GPU available: True (mps), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


/Users/f.costa/Code/NodeField/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/f.costa/Code/NodeField/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=17` in the `DataLoader` to improve performance.
/Users/f.costa/Code/NodeField/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=17` in the `DataLoader` to improve performance.


/Users/f.costa/Code/NodeField/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


...

💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


/Users/f.costa/Code/NodeField/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/f.costa/Code/NodeField/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=17` in the `DataLoader` to improve performance.
/Users/f.costa/Code/NodeField/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=17` in the `DataLoader` to improve performance.


/Users/f.costa/Code/NodeField/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


`Trainer.fit` stopped: `max_epochs=10` reached.


Trained recurrent_energy_annealed seed=0 updates=50


## 6 — Fixed-depth comparison and recorded evaluation
Runs the selected smoke/full matrix. Failed decodes remain rows. Unaligned generated graphs use label distributions rather than node-wise label accuracy.

In [7]:
experiment.evaluate()
summary = summarize_results(experiment.run_dir)
results, diagnostics = load_results(experiment.run_dir)
analysis_section(results, diagnostics, "fixed_depth", k_train=K_TRAIN)


Evaluated baseline seed=0 depth=1


Evaluated baseline seed=0 depth=2


Evaluated baseline seed=0 depth=4


Evaluated baseline seed=0 depth=8


Evaluated baseline seed=0 depth=16


...

Evaluated recurrent_energy_annealed seed=0 depth=4


Evaluated recurrent_energy_annealed seed=0 depth=8


Evaluated recurrent_energy_annealed seed=0 depth=16


Evaluated recurrent_energy_annealed seed=0 depth=32


,seed,K_train,K_test,intervention_step,example_id,sampling_seed,valid,decoder_success,feasible_condition_match,node_count_accuracy,...,feasibility_violations,runtime_seconds,generation_seconds,decode_seconds,diagnostic_seconds,field_evaluations,diagnostic_readouts,final_readout_field_evaluations,peak_gpu_memory,failure
model,,,,,,,,,,,,,,,,,,,,,
baseline,0.0,1.0,8.0,NaN,39.1,1039.1,1.0,1.0,0.0,1.0,...,NaN,0.101263,0.006343,0.094919,0.0,8.0,0.0,1.0,NaN,NaN
recurrent_energy_annealed,0.0,8.0,8.0,NaN,39.1,1039.1,1.0,1.0,0.0,1.0,...,NaN,0.089328,0.010206,0.079122,0.0,8.0,8.0,0.0,NaN,NaN
recurrent_energy_constant,0.0,8.0,8.0,NaN,39.1,1039.1,1.0,1.0,0.0,1.0,...,NaN,0.080843,0.010512,0.070332,0.0,8.0,8.0,0.0,NaN,NaN


## 7 — Inference-depth scaling
The same checkpoints are evaluated beyond training depth. Full depths extend to 256; stop on nonfinite states.

In [8]:
analysis_section(results, diagnostics, "depth", run_dir=experiment.run_dir, k_train=K_TRAIN)


seed  K_train  intervention_step  \
model                     K_test                                     
baseline                  1        0.0      1.0                NaN   
                          2        0.0      1.0                NaN   
                          4        0.0      1.0                NaN   
                          8        0.0      1.0                NaN   
                          16       0.0      1.0                NaN   
                          32       0.0      1.0                NaN   
recurrent_energy_annealed 1        0.0      8.0                NaN   
                          2        0.0      8.0                NaN   
                          4        0.0      8.0                NaN   
                          8        0.0      8.0                NaN   
                          16       0.0      8.0                NaN   
                          32       0.0      8.0                NaN   
recurrent_energy_constant 1        0.0      8.0                NaN   
                          2        0.0      8.0                NaN   
                          4        0.0      8.0                NaN   
                          8        0.0      8.0                NaN   
                          16       0.0      8.0                NaN   
                          32       0.0      8.0                NaN   

                                  example_id  sampling_seed  valid  \
model                     K_test                                     
baseline                  1             39.1         1039.1    1.0   
                          2             39.1         1039.1    1.0   
                          4             39.1         1039.1    1.0   
                          8             39.1         1039.1    1.0   
                          16            39.1         1039.1    1.0   
                          32            39.1         1039.1    1.0   
recurrent_energy_annealed 1             39.1         1039.1    1.0   
                          2             39.1         1039.1    1.0   
                          4             39.1         1039.1    1.0   
                          8             39.1         1039.1    1.0   
                          16            39.1         1039.1    1.0   
                          32            39.1         1039.1    1.0   
recurrent_energy_constant 1             39.1         1039.1    1.0   
                          2             39.1         1039.1    1.0   
                          4             39.1         1039.1    1.0   
                          8             39.1         1039.1    1.0   
                          16            39.1         1039.1    1.0   
                          32            39.1         1039.1    1.0   

                                  decoder_success  feasible_condition_match  \
model                     K_test                                              
baseline                  1                   1.0                       0.0   
                          2                   1.0                       0.0   
                          4                   1.0                       0.0   
                          8                   1.0                       0.0   
                          16                  1.0                       0.0   
                          32                  1.0                       0.0   
recurrent_energy_annealed 1                   1.0                       0.0   
                          2                   1.0                       0.0   
                          4                   1.0                       0.0   
                          8                   1.0                       0.0   
                          16                  1.0                       0.0   
                          32                  1.0                       0.0   
recurrent_energy_constant 1                   1.0                       0.0   
                          2                   1

## 8 — Hidden-state reset
Reset occurs before the zero-based designated evaluation. Compare head count errors and saved per-step quality around the intervention.

In [9]:
analysis_section(results, diagnostics, "reset", run_dir=experiment.run_dir, k_train=K_TRAIN)


,seed,checkpoint,model,model_mode,training_schedule,K_train,K_test,intervention,intervention_step,inference_noise,...,readout_valid,readout_decoder_success,readout_feasible_condition_match,readout_node_count_accuracy,readout_edge_count_accuracy,readout_condition_error,readout_degree_consistency,readout_node_label_distribution_error,readout_edge_label_distribution_error,readout_feasibility_violations
0,0,best,recurrent_energy_constant,recurrent_energy,constant,8,1,normal,NaN,none,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,best,recurrent_energy_constant,recurrent_energy,constant,8,1,normal,NaN,none,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,best,recurrent_energy_constant,recurrent_energy,constant,8,1,normal,NaN,none,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,best,recurrent_energy_constant,recurrent_energy,constant,8,1,normal,NaN,none,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,best,recurrent_energy_constant,recurrent_energy,constant,8,1,normal,NaN,none,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4395,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,reset_h_mid,16.0,none,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4396,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,reset_h_mid,16.0,none,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4397,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,reset_h_mid,16.0,none,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4398,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,reset_h_mid,16.0,none,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 9 — Hidden-state shuffle
Full mode runs three independent within-graph permutations at each reset fraction. Smoke mode leaves this analysis empty.

In [10]:
analysis_section(results, diagnostics, "shuffle", run_dir=experiment.run_dir, k_train=K_TRAIN)


,seed,checkpoint,model,model_mode,training_schedule,K_train,K_test,intervention,intervention_step,inference_noise,...,feasibility_violations,runtime_seconds,generation_seconds,decode_seconds,diagnostic_seconds,field_evaluations,diagnostic_readouts,final_readout_field_evaluations,peak_gpu_memory,failure
0,0,best,baseline,baseline,baseline_fixed,1,1,normal,NaN,none,...,NaN,0.849395,0.014439,0.834955,0.0,1,0,1,NaN,NaN
1,0,best,baseline,baseline,baseline_fixed,1,1,normal,NaN,none,...,NaN,0.130416,0.004260,0.126156,0.0,1,0,1,NaN,NaN
2,0,best,baseline,baseline,baseline_fixed,1,1,normal,NaN,none,...,NaN,0.081525,0.004064,0.077461,0.0,1,0,1,NaN,NaN
3,0,best,baseline,baseline,baseline_fixed,1,1,normal,NaN,none,...,NaN,0.136814,0.003829,0.132985,0.0,1,0,1,NaN,NaN
4,0,best,baseline,baseline,baseline_fixed,1,1,normal,NaN,none,...,NaN,0.070437,0.003916,0.066521,0.0,1,0,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
515,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,reset_h_mid,16.0,none,...,NaN,0.115921,0.028147,0.087774,0.0,32,32,0,NaN,NaN
516,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,reset_h_mid,16.0,none,...,NaN,0.087634,0.026953,0.060681,0.0,32,32,0,NaN,NaN
517,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,reset_h_mid,16.0,none,...,NaN,0.099420,0.027307,0.072112,0.0,32,32,0,NaN,NaN
518,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,reset_h_mid,16.0,none,...,NaN,0.114684,0.027671,0.087013,0.0,32,32,0,NaN,NaN


## 10 — Observable-state destruction
Full mode includes the complete persistent/fresh-x × persistent/reset-h grid. The smoke matrix contains normal, midpoint resets, and fresh-x every step.

In [11]:
analysis_section(results, diagnostics, "channels", run_dir=experiment.run_dir, k_train=K_TRAIN)


,seed,checkpoint,model,model_mode,training_schedule,K_train,K_test,intervention,intervention_step,inference_noise,...,feasibility_violations,runtime_seconds,generation_seconds,decode_seconds,diagnostic_seconds,field_evaluations,diagnostic_readouts,final_readout_field_evaluations,peak_gpu_memory,failure
0,0,best,baseline,baseline,baseline_fixed,1,1,normal,NaN,none,...,NaN,0.849395,0.014439,0.834955,0.0,1,0,1,NaN,NaN
1,0,best,baseline,baseline,baseline_fixed,1,1,normal,NaN,none,...,NaN,0.130416,0.004260,0.126156,0.0,1,0,1,NaN,NaN
2,0,best,baseline,baseline,baseline_fixed,1,1,normal,NaN,none,...,NaN,0.081525,0.004064,0.077461,0.0,1,0,1,NaN,NaN
3,0,best,baseline,baseline,baseline_fixed,1,1,normal,NaN,none,...,NaN,0.136814,0.003829,0.132985,0.0,1,0,1,NaN,NaN
4,0,best,baseline,baseline,baseline_fixed,1,1,normal,NaN,none,...,NaN,0.070437,0.003916,0.066521,0.0,1,0,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,fresh_x_every_step,NaN,unit_gaussian,...,NaN,0.123272,0.028598,0.094674,0.0,32,32,0,NaN,NaN
536,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,fresh_x_every_step,NaN,unit_gaussian,...,NaN,0.096481,0.028148,0.068333,0.0,32,32,0,NaN,NaN
537,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,fresh_x_every_step,NaN,unit_gaussian,...,NaN,0.094573,0.030471,0.064102,0.0,32,32,0,NaN,NaN
538,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,fresh_x_every_step,NaN,unit_gaussian,...,NaN,0.144725,0.028501,0.116223,0.0,32,32,0,NaN,NaN


## 11 — Training curriculum
Compare constant and annealed models under normal inference, fresh-x noise, and increased depth. Full mode also compares retained checkpoints at matched updates.

In [12]:
analysis_section(results, diagnostics, "curriculum", run_dir=experiment.run_dir, k_train=K_TRAIN)


seed  K_train  \
training_schedule checkpoint K_test intervention                        
annealed          best       1      fresh_x_every_step   0.0      8.0   
                                    fresh_x_mid          0.0      8.0   
                                    normal               0.0      8.0   
                                    reset_h_mid          0.0      8.0   
                             2      fresh_x_every_step   0.0      8.0   
                                    fresh_x_mid          0.0      8.0   
                                    normal               0.0      8.0   
                                    reset_h_mid          0.0      8.0   
                             4      fresh_x_every_step   0.0      8.0   
                                    fresh_x_mid          0.0      8.0   
                                    normal               0.0      8.0   
                                    reset_h_mid          0.0      8.0   
                             8      fresh_x_every_step   0.0      8.0   
                                    fresh_x_mid          0.0      8.0   
                                    normal               0.0      8.0   
                                    reset_h_mid          0.0      8.0   
                             16     fresh_x_every_step   0.0      8.0   
                                    fresh_x_mid          0.0      8.0   
                                    normal               0.0      8.0   
                                    reset_h_mid          0.0      8.0   
                             32     fresh_x_every_step   0.0      8.0   
                                    fresh_x_mid          0.0      8.0   
                                    normal               0.0      8.0   
                                    reset_h_mid          0.0      8.0   
constant          best       1      fresh_x_every_step   0.0      8.0   
                                    fresh_x_mid          0.0      8.0   
                                    normal               0.0      8.0   
                                    reset_h_mid          0.0      8.0   
                             2      fresh_x_every_step   0.0      8.0   
                                    fresh_x_mid          0.0      8.0   
                                    normal               0.0      8.0   
                                    reset_h_mid          0.0      8.0   
                             4      fresh_x_every_step   0.0      8.0   
                                    fresh_x_mid          0.0      8.0   
                                    normal               0.0      8.0   
                                    reset_h_mid          0.0      8.0   
                             8      fresh_x_every_step   0.0      8.0   
                                    fresh_x_mid          0.0      8.0   
                                    normal               0.0      8.0   
                                    reset_h_mid          0.0      8.0   
                             16     fresh_x_every_step   0.0      8.0   
                                    fresh_x_mid          0.0      8.0   
                                    normal               0.0      8.0   
                                    reset_h_mid          0.0      8.0   
                             32     fresh_x_every_step   0.0      8.0   
                                    fresh_x_mid          0.0      8.0   
                                    normal               0.0      8.0   
                                    reset_h_mid          0.0      8.0   

                                                        intervention_step  \
training_schedule checkpoint K_test intervention                            
annealed          best       1      fresh_x_every_step                NaN   
                                    fresh_x_mid                       0.0   
                                    normal                            NaN   
                    

## 12 — Memory versus repeated computation
Full mode resets h before every evaluation on the same trained checkpoint.

In [13]:
analysis_section(results, diagnostics, "no_memory", run_dir=experiment.run_dir, k_train=K_TRAIN)


,seed,checkpoint,model,model_mode,training_schedule,K_train,K_test,intervention,intervention_step,inference_noise,...,feasibility_violations,runtime_seconds,generation_seconds,decode_seconds,diagnostic_seconds,field_evaluations,diagnostic_readouts,final_readout_field_evaluations,peak_gpu_memory,failure
0,0,best,baseline,baseline,baseline_fixed,1,1,normal,NaN,none,...,NaN,0.849395,0.014439,0.834955,0.0,1,0,1,NaN,NaN
1,0,best,baseline,baseline,baseline_fixed,1,1,normal,NaN,none,...,NaN,0.130416,0.004260,0.126156,0.0,1,0,1,NaN,NaN
2,0,best,baseline,baseline,baseline_fixed,1,1,normal,NaN,none,...,NaN,0.081525,0.004064,0.077461,0.0,1,0,1,NaN,NaN
3,0,best,baseline,baseline,baseline_fixed,1,1,normal,NaN,none,...,NaN,0.136814,0.003829,0.132985,0.0,1,0,1,NaN,NaN
4,0,best,baseline,baseline,baseline_fixed,1,1,normal,NaN,none,...,NaN,0.070437,0.003916,0.066521,0.0,1,0,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
505,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,normal,NaN,none,...,NaN,0.093897,0.028368,0.065528,0.0,32,32,0,NaN,NaN
506,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,normal,NaN,none,...,NaN,0.086085,0.027768,0.058317,0.0,32,32,0,NaN,NaN
507,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,normal,NaN,none,...,NaN,0.089864,0.027613,0.062251,0.0,32,32,0,NaN,NaN
508,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,normal,NaN,none,...,NaN,0.107340,0.027570,0.079770,0.0,32,32,0,NaN,NaN


## 13 — Inference noise
Fresh x is N(0, s²I), with s recorded explicitly. Replacement noise and Langevin noise use distinct controls. All inference x values use the model’s scaled feature space.

In [14]:
analysis_section(results, diagnostics, "noise", run_dir=experiment.run_dir, k_train=K_TRAIN)


seed  K_train  \
model                     inference_noise K_test                  
baseline                  none            1        0.0      1.0   
                                          2        0.0      1.0   
                                          4        0.0      1.0   
                                          8        0.0      1.0   
                                          16       0.0      1.0   
                                          32       0.0      1.0   
recurrent_energy_annealed none            1        0.0      8.0   
                                          2        0.0      8.0   
                                          4        0.0      8.0   
                                          8        0.0      8.0   
                                          16       0.0      8.0   
                                          32       0.0      8.0   
                          unit_gaussian   1        0.0      8.0   
                                          2        0.0      8.0   
                                          4        0.0      8.0   
                                          8        0.0      8.0   
                                          16       0.0      8.0   
                                          32       0.0      8.0   
recurrent_energy_constant none            1        0.0      8.0   
                                          2        0.0      8.0   
                                          4        0.0      8.0   
                                          8        0.0      8.0   
                                          16       0.0      8.0   
                                          32       0.0      8.0   
                          unit_gaussian   1        0.0      8.0   
                                          2        0.0      8.0   
                                          4        0.0      8.0   
                                          8        0.0      8.0   
                                          16       0.0      8.0   
                                          32       0.0      8.0   

                                                  intervention_step  \
model                     inference_noise K_test                      
baseline                  none            1                     NaN   
                                          2                     NaN   
                                          4                     NaN   
                                          8                     NaN   
                                          16                    NaN   
                                          32                    NaN   
recurrent_energy_annealed none            1                     0.0   
                                          2                     1.0   
                                          4                     2.0   
                                          8                     4.0   
                                          16                    8.0   
                                          32                   16.0   
                          unit_gaussian   1                     0.0   
                                          2                     1.0   
                                          4                     2.0   
                                          8                     4.0   
                                          16                    8.0   
                                          32                   16.0   
recurrent_energy_constant none            1                     0.0   
                                          2                     1.0   
                                          4                     2.0   
                                          8                     4.0   
                                          16                    8.0   
                                          32                   16.0   
                          unit_gaussian   1                

## 14 — State stability
Inspect hidden delta, score norm, potential, and prediction changes. These are empirical diagnostics; potential need not decrease when memory changes.

In [15]:
analysis_section(results, diagnostics, "stability", run_dir=experiment.run_dir, k_train=K_TRAIN)


,seed,checkpoint,model,model_mode,training_schedule,K_train,K_test,intervention,intervention_step,inference_noise,...,readout_valid,readout_decoder_success,readout_feasible_condition_match,readout_node_count_accuracy,readout_edge_count_accuracy,readout_condition_error,readout_degree_consistency,readout_node_label_distribution_error,readout_edge_label_distribution_error,readout_feasibility_violations
0,0,best,recurrent_energy_constant,recurrent_energy,constant,8,1,normal,NaN,none,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,best,recurrent_energy_constant,recurrent_energy,constant,8,1,normal,NaN,none,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,best,recurrent_energy_constant,recurrent_energy,constant,8,1,normal,NaN,none,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,best,recurrent_energy_constant,recurrent_energy,constant,8,1,normal,NaN,none,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,best,recurrent_energy_constant,recurrent_energy,constant,8,1,normal,NaN,none,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5035,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,fresh_x_every_step,NaN,unit_gaussian,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5036,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,fresh_x_every_step,NaN,unit_gaussian,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5037,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,fresh_x_every_step,NaN,unit_gaussian,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5038,0,best,recurrent_energy_annealed,recurrent_energy,annealed,8,32,fresh_x_every_step,NaN,unit_gaussian,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 15 — Anytime computation
Full mode selects stopping thresholds using validation trajectories, then measures test steps saved and mean/worst quality loss. Decoder-unchanged stopping uses three consecutive unchanged solutions. This secondary experiment is not executed in smoke mode.

In [16]:
analysis_section(results, diagnostics, "anytime", run_dir=experiment.run_dir, k_train=K_TRAIN)


'Prepared for full run; not executed in smoke mode.'

## 16 — Statistical tables
Full comparisons aggregate independent training seeds and use paired seed-level confidence intervals. A one-seed run has undefined across-seed standard deviations and confidence intervals.

In [17]:
analysis_section(results, diagnostics, "statistics", run_dir=experiment.run_dir, k_train=K_TRAIN)


,model,checkpoint,training_schedule,K_test,intervention,inference_noise,metric,mean,std,ci_low,ci_high,seeds
0,baseline,best,baseline_fixed,1,normal,none,feasible_condition_match,0.000000,NaN,NaN,NaN,1
1,baseline,best,baseline_fixed,1,normal,none,valid,1.000000,NaN,NaN,NaN,1
2,baseline,best,baseline_fixed,1,normal,none,decoder_success,1.000000,NaN,NaN,NaN,1
3,baseline,best,baseline_fixed,1,normal,none,node_count_accuracy,1.000000,NaN,NaN,NaN,1
4,baseline,best,baseline_fixed,1,normal,none,edge_count_accuracy,1.000000,NaN,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...
427,recurrent_energy_constant,best,constant,32,reset_h_mid,none,node_count_accuracy,1.000000,NaN,NaN,NaN,1
428,recurrent_energy_constant,best,constant,32,reset_h_mid,none,edge_count_accuracy,1.000000,NaN,NaN,NaN,1
429,recurrent_energy_constant,best,constant,32,reset_h_mid,none,condition_error,0.655556,NaN,NaN,NaN,1
430,recurrent_energy_constant,best,constant,32,reset_h_mid,none,generation_seconds,0.027183,NaN,NaN,NaN,1


## 17 — Figures
All values come from saved results, diagnostics and training history. Re-run this cell to regenerate the figures without retraining.

In [18]:
plot_results(experiment.run_dir)


[PosixPath('artifact/recurrent_nodefield/20260905T111715-2f96c4ec/figures/01_architecture.png'),
 PosixPath('artifact/recurrent_nodefield/20260905T111715-2f96c4ec/figures/02_condition_error.png'),
 PosixPath('artifact/recurrent_nodefield/20260905T111715-2f96c4ec/figures/02_decoder_success.png'),
 PosixPath('artifact/recurrent_nodefield/20260905T111715-2f96c4ec/figures/02_quality_depth.png'),
 PosixPath('artifact/recurrent_nodefield/20260905T111715-2f96c4ec/figures/03_memory_interventions.png'),
 PosixPath('artifact/recurrent_nodefield/20260905T111715-2f96c4ec/figures/03_memory_recovery.png'),
 PosixPath('artifact/recurrent_nodefield/20260905T111715-2f96c4ec/figures/04_information_channels.png'),
 PosixPath('artifact/recurrent_nodefield/20260905T111715-2f96c4ec/figures/05_state_dynamics.png'),
 PosixPath('artifact/recurrent_nodefield/20260905T111715-2f96c4ec/figures/training_score.png'),
 PosixPath('artifact/recurrent_nodefield/20260905T111715-2f96c4ec/figures/training_structural.png'),

## 18 — Decision criteria
The predefined effect criterion is applied only to full, independent-seed comparisons. Memory intervention evidence remains an intervention-based interpretation. Preservation under fresh-x corruption is an open experimental question, not an expected result.

In [19]:
decision_report(experiment.run_dir)


{'scope': 'smoke',
 'conclusion': 'Numerical and workflow checks only; no significance or mechanism claim from one seed.'}